# 🤖 BKC AI Workshop — สร้าง Mini AI Agent ด้วย Microsoft Foundry

**วิธีใช้โน้ตบุ๊กนี้:** เลื่อนลงทีละหัวข้อ อ่านคำอธิบาย แล้วกดปุ่ม **▶ Run** (หรือ `Shift+Enter`) ที่มุมซ้ายของแต่ละกล่องโค้ด **ไล่จากบนลงล่างตามลำดับ**

> 💡 ช่องที่ให้ "แก้เอง" จะมีเครื่องหมาย `# ✏️ แก้ได้ตรงนี้` กำกับไว้ — จุดอื่นกด Run ได้เลย ไม่ต้องแก้
>
> ⚠️ **ห้ามใส่ข้อมูลลับ** (ราคา, ข้อมูลลูกค้า, สูตรการผลิต, ข้อมูลส่วนบุคคล) — ใช้ข้อมูลตัวอย่างเท่านั้น
>
> 🙋 ติดตรงไหน **ยกมือเรียก TA ได้เลย**


---
## 0️⃣ เตรียมเครื่อง (ทำครั้งเดียวตอนเริ่ม)

รัน 3 กล่องด้านล่างตามลำดับ: **ติดตั้งไลบรารี → ใส่กุญแจจากผู้สอน → ทดสอบการเชื่อมต่อ**


In [ ]:
# ติดตั้งไลบรารีที่ต้องใช้ (รอสักครู่จนขึ้น ✅)
!pip install -q openai openpyxl pandas ipywidgets
print("✅ ติดตั้งเรียบร้อย ไปกล่องถัดไปได้เลย")


In [ ]:
#@title 🔑 ใส่ข้อมูลจากผู้สอน (กรอกช่อง แล้วกด Run) { display-mode: "form" }
AZURE_OPENAI_API_KEY=""
AZURE_OPENAI_ENDPOINT=""
AZURE_OPENAI_DEPLOYMENT="gpt-5-mini"
AZURE_OPENAI_API_VERSION="2024-02-15-preview"

print("บันทึกข้อมูลเรียบร้อย ✅ ไปกล่องถัดไปเพื่อทดสอบการเชื่อมต่อ")


บันทึกข้อมูลเรียบร้อย ✅ ไปกล่องถัดไปเพื่อทดสอบการเชื่อมต่อ


In [15]:
# สร้างการเชื่อมต่อกับ AI แล้วทดสอบว่าคุยได้จริง
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)
deployment_name = AZURE_OPENAI_DEPLOYMENT

try:
    test = client.chat.completions.create(
        model=deployment_name,
        messages=[{"role": "user", "content": "ตอบกลับสั้น ๆ ว่า 'พร้อมแล้ว'"}],
    )
    print("✅ เชื่อมต่อ AI สำเร็จ:", test.choices[0].message.content)
except Exception as e:
    print("❌ เชื่อมต่อไม่สำเร็จ — ยกมือเรียก TA ได้เลย")
    print("รายละเอียด:", e)


✅ เชื่อมต่อ AI สำเร็จ: พร้อมแล้ว


---
## 1️⃣ Lab 1 — คุยกับ AI ครั้งแรกผ่านโค้ด

**แนวคิด:** โปรแกรมของเราส่งข้อความ (prompt) ไปที่ Endpoint ของ Foundry แล้ว AI ส่งคำตอบกลับมา — นี่คือหัวใจของการ "ฝัง AI ในแอป"

พิมพ์คำถามอะไรก็ได้ในช่อง `MY_QUESTION` แล้วกด Run


In [16]:
# ✏️ แก้ได้ตรงนี้จุดเดียว: เปลี่ยนคำถามที่อยากถาม AI
MY_QUESTION = "ช่วยแนะนำวิธีดูแลมอเตอร์ปั๊มน้ำ 3 ข้อสั้น ๆ"

try:
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. ตอบเป็นภาษาไทย"},
            {"role": "user", "content": MY_QUESTION},
        ],
    )
    print("🤖 คำตอบของ AI:\n")
    print(response.choices[0].message.content)
except Exception as e:
    print("⚠️ AI แน่นชั่วคราว รอ 5 วินาทีแล้วกด Run ใหม่ (ถ้ายังไม่ได้ เรียก TA)")
    print("รายละเอียด:", e)


🤖 คำตอบของ AI:

1) ตรวจสภาพและความสะอาดสม่ำเสมอ — ล้างกรอง/ท่อ ขจัดเศษและพ่นฝุ่นรอบมอเตอร์  
2) หล่อลื่นและสังเกตอาการผิดปกติ — เติมจาระบี/เปลี่ยนลูกปืนตามคู่มือ หากมีสั่นหรือเสียงดังให้หยุดตรวจทันที  
3) ตรวจระบบไฟฟ้าและการติดตั้ง — เช็คสายต่อ ขั้วและการต่อกราวด์ ห้ามซ่อมขณะมีไฟ ตัดไฟก่อนทุกครั้ง


---
## 2️⃣ Lab 2 — ให้ AI ตอบเป็น JSON (โครงสร้างที่โปรแกรมอ่านต่อได้)

**แนวคิด:** ถ้า AI ตอบเป็นข้อความยาว ๆ โปรแกรมเอาไปใช้ต่อยาก แต่ถ้าบังคับให้ตอบเป็น **JSON** เราจะดึงค่าทีละช่อง (หมวดหมู่, ความเร่งด่วน) ไปทำสี ทำการ์ด หรือใส่ Excel ได้ทันที

ลองเปลี่ยนข้อความปัญหาในช่อง `ISSUE_TEXT` เป็นเคสอื่นดู


In [17]:
import json

# ✏️ แก้ได้ตรงนี้จุดเดียว: ใส่ข้อความปัญหาที่อยากให้ AI วิเคราะห์
ISSUE_TEXT = "มอเตอร์ปั๊มน้ำหล่อเย็นไลน์ 2 มีเสียงดังผิดปกติและมีควันที่ขั้วต่อสายไฟ ยังไม่ได้ปิดเครื่อง"

prompt = f"""
คุณเป็น AI ผู้ช่วยวิเคราะห์ปัญหาในโรงงานของ BKC
วิเคราะห์ข้อความปัญหาด้านล่าง แล้วตอบกลับเป็น JSON เท่านั้น

กติกา:
- จัดหมวดหมู่ (category): Mechanical, Electrical, Quality, Safety หรือ Other
- กำหนดความเร่งด่วน (priority): Low, Medium หรือ High
- ห้ามเดาข้อมูลที่ไม่มี

ตอบเป็น JSON ที่มี field: summary, category, priority, recommended_action

ข้อความปัญหา:
{ISSUE_TEXT}
"""

try:
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You return valid JSON only."},
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)

    print("📋 ผลวิเคราะห์จาก AI")
    print("-" * 40)
    print("สรุป        :", result.get("summary"))
    print("หมวดหมู่     :", result.get("category"))
    print("ความเร่งด่วน :", result.get("priority"))
    print("ควรทำ       :", result.get("recommended_action"))

    if result.get("priority") == "High":
        print("\n🚨 ความเร่งด่วนสูง — ควรแจ้งหัวหน้างานทันที")
except Exception as e:
    print("⚠️ เกิดข้อผิดพลาด รอ 5 วิแล้วกด Run ใหม่ หรือเรียก TA")
    print("รายละเอียด:", e)


📋 ผลวิเคราะห์จาก AI
----------------------------------------
สรุป        : มอเตอร์ปั๊มน้ำหล่อเย็น (ไลน์ 2) ส่งเสียงดังผิดปกติและมีควันที่บริเวณขั้วต่อสายไฟ ขณะเครื่องยังไม่ได้ปิด/ยังมีพลังงานจ่ายอยู่
หมวดหมู่     : Electrical
ความเร่งด่วน : High
ควรทำ       : การดำเนินการทันที: 1) หยุดเครื่องและตัดไฟของมอเตอร์โดยทันทีตามขั้นตอนความปลอดภัยของโรงงาน (Lockout-Tagout) — อย่าพยายามสัมผัสขั้วต่อขณะที่มีพลังงาน 2) หากมีควันหนา/ไฟ ให้แจ้งทีมดับเพลิงในโรงงานและอพยพบริเวณใกล้เคียงตามแผนฉุกเฉิน 3) แจ้งผู้ควบคุม/หัวหน้างานและทีมช่างไฟฟ้าทันที การตรวจสอบเบื้องต้น (หลังตัดไฟและ LOTO): 4) ตรวจสอบความเสียหายที่ขั้วต่อ สายไฟ และฉนวนสาย (ภาพถ่าย/บันทึกไว้) 5) ใช้เครื่องมือวัดความต่อเนื่อง/ความต้านทานฉนวนและตรวจสอบการต่อสายและแรงยึดของขั้วต่อ 6) ตรวจสอบความร้อนสะสมด้วยกล้องความร้อน (ถ้ามีบันทึกก่อนปัญหาเกิดซ้ำ) 7) ตรวจสอบสาเหตุของเสียงดังหลังการทำให้ปลอดพลังงาน เช่น ลูกปืน เสื้อลูกปืน โรเตอร์/แบริ่ง หรือการติดขัดของปั๊ม โดยช่างกล (ไม่สันนิษฐานก่อนตรวจ) การซ่อมและการกู้คืน: 8) เปลี่ยนหรือซ่อมขั้วต่อ สายไฟ

---
## 3️⃣ Lab 3 — Human-in-the-Loop (คนตรวจก่อนเสมอ)

> 👀 **ส่วนนี้ผู้สอนจะ demo หน้าเว็บ Streamlit ให้ดูบนจอ** (เวอร์ชันเต็มอยู่ในโฟลเดอร์ `workshops/lab-03` สำหรับคนที่อยากลองทำบนเครื่องตัวเอง)

ด้านล่างคือเวอร์ชันย่อที่ทำได้ใน Colab เลย: **AI วิเคราะห์มาให้ก่อน → คนแก้ไขในกล่อง → กดยืนยัน** นี่คือหลักการสำคัญ — *ไม่ปล่อยให้ AI ตัดสินใจเอง 100% งานที่เกี่ยวกับความปลอดภัย/สายการผลิตต้องมีคนตรวจ*


In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display

# ✏️ แก้ได้ตรงนี้: ข้อความปัญหาที่จะให้ AI ช่วยกรอกร่างให้ก่อน
ISSUE_TEXT = "เซ็นเซอร์อุณหภูมิเตาอบเบอร์ 3 แสดง 150 องศา ทั้งที่ตั้งไว้ 120 องศา ชิ้นงานเริ่มไหม้"

# ให้ AI วิเคราะห์เพื่อ "เติมร่าง" (pre-fill)
try:
    r = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You return valid JSON only."},
            {"role": "user", "content": f"วิเคราะห์ปัญหานี้ ตอบ JSON field: category, priority, recommended_action.\n\n{ISSUE_TEXT}"},
        ],
        response_format={"type": "json_object"},
    )
    ai = json.loads(r.choices[0].message.content)
except Exception as e:
    ai = {"category": "", "priority": "", "recommended_action": ""}
    print("⚠️ ดึงผล AI ไม่ได้ กรอกเองได้เลย:", e)

# กล่องให้ "คน" ตรวจและแก้ไขก่อนยืนยัน
w_cat = widgets.Text(value=ai.get("category", ""), description="หมวดหมู่:")
w_pri = widgets.Dropdown(options=["Low", "Medium", "High"],
                         value=ai.get("priority") if ai.get("priority") in ["Low","Medium","High"] else "Medium",
                         description="ความเร่งด่วน:")
w_act = widgets.Textarea(value=ai.get("recommended_action", ""), description="ควรทำ:",
                         layout=widgets.Layout(width="600px", height="80px"))
btn = widgets.Button(description="✅ ยืนยันบันทึก", button_style="success")
out = widgets.Output()

def on_click(_):
    with out:
        out.clear_output()
        print("💾 บันทึกเข้าระบบแล้ว (ผ่านการตรวจโดยพนักงาน):")
        print(" หมวดหมู่    :", w_cat.value)
        print(" ความเร่งด่วน:", w_pri.value)
        print(" ควรทำ       :", w_act.value)

btn.on_click(on_click)
print("🤖 AI ร่างให้แล้ว — ตรวจ/แก้ได้ตามต้องการ แล้วกดยืนยัน")
display(w_cat, w_pri, w_act, btn, out)


🤖 AI ร่างให้แล้ว — ตรวจ/แก้ได้ตามต้องการ แล้วกดยืนยัน


Text(value='ความผิดปกติของเซ็นเซอร์/ระบบควบคุมอุณหภูมิ (การอ่านผิดพลาดหรืออุณหภูมิเกิน)', description='หมวดหมู…

Dropdown(description='ความเร่งด่วน:', index=1, options=('Low', 'Medium', 'High'), value='Medium')

Textarea(value='1) หยุดการทำงานของเตา/ตัดพลังงานระบบทำความร้อนทันทีเพื่อลดความเสี่ยงต่อการลุกไหม้; 2) นำชิ้นงา…

Button(button_style='success', description='✅ ยืนยันบันทึก', style=ButtonStyle())

Output()

---
## 4️⃣ Lab 4 — ประมวลผลหลายเคสพร้อมกัน แล้วบันทึกลง Excel

**แนวคิด:** งานจริงมีปัญหาเป็นร้อยเป็นพันแถว เราวนลูปให้ AI วิเคราะห์ทีละแถว แล้วเขียนผลกลับลง Excel — พร้อม `try-except` เพื่อว่า **ถ้าแถวไหนพัง โปรแกรมข้ามไปทำแถวต่อไป ไม่ล่มทั้งไฟล์**

กด Run แล้วรอสักครู่ ระบบจะให้ดาวน์โหลดไฟล์ `.xlsx` ตอนจบ


In [19]:
import json
import pandas as pd
from google.colab import files

# ข้อมูลตัวอย่าง (งานจริงจะอ่านจากไฟล์ Excel/CSV ที่อัปโหลด)
issues = [
    {"case_id": "FAC-001", "issue_report": "มอเตอร์ปั๊มน้ำหล่อเย็นไลน์ 2 มีเสียงดังและมีควันที่ขั้วต่อสายไฟ"},
    {"case_id": "FAC-002", "issue_report": "พบคราบน้ำมันบนผิวชิ้นงานพลาสติกโมลด์เบอร์ 4 เกินมาตรฐาน 2%"},
    {"case_id": "FAC-003", "issue_report": "สายไฟตู้เชื่อมชำรุดเห็นลวดทองแดง วางพาดทางเดิน ยังไม่มีป้ายเตือน"},
    {"case_id": "FAC-004", "issue_report": "เซ็นเซอร์อุณหภูมิเตาอบเบอร์ 3 แสดง 150C ทั้งที่ตั้ง 120C ชิ้นงานเริ่มไหม้"},
    {"case_id": "FAC-005", "issue_report": "โฟล์คลิฟต์เบอร์ 02 ชนชั้นวางฝั่ง C น้ำยาเคมีหก 2 ลิตร"},
]

def analyze(text):
    r = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You return valid JSON only."},
            {"role": "user", "content": f"วิเคราะห์ปัญหานี้ ตอบ JSON field: category (Mechanical/Electrical/Quality/Safety/Other), priority (Low/Medium/High), summary.\n\nปัญหา: {text}"},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(r.choices[0].message.content)

rows = []
for item in issues:
    try:
        a = analyze(item["issue_report"])
        rows.append({"case_id": item["case_id"], "issue": item["issue_report"],
                     "category": a.get("category", "-"), "priority": a.get("priority", "-"),
                     "summary": a.get("summary", "-")})
        print(f'✅ {item["case_id"]} -> {a.get("category")} / {a.get("priority")}')
    except Exception as e:
        rows.append({"case_id": item["case_id"], "issue": item["issue_report"],
                     "category": "Error", "priority": "Error", "summary": str(e)})
        print(f'❌ {item["case_id"]} พัง — ข้ามไปทำแถวต่อไป')

df = pd.DataFrame(rows)
df.to_excel("factory_issues_result.xlsx", index=False)
print("\n📁 สร้างไฟล์ factory_issues_result.xlsx แล้ว กำลังให้ดาวน์โหลด...")
display(df)
files.download("factory_issues_result.xlsx")


ModuleNotFoundError: No module named 'google.colab'

---
## 5️⃣ Mini Challenge — ทำของแผนกคุณเอง 🎯

ลองเอาที่เรียนมาปรับกับงานจริงในแผนกของคุณ (QA, ซ่อมบำรุง, ความปลอดภัย, จัดซื้อ, HR ...)

**แก้แค่ 2 จุด:** (1) กติกา/หมวดหมู่ใน `PROMPT` ให้เข้ากับงานคุณ  (2) `INPUT_TEXT` ข้อความงานจริง (ไม่ลับ)


In [ ]:
import json

# ✏️ จุดที่ 1: ใส่ข้อความงานจริงของคุณ (ห้ามใส่ข้อมูลลับ)
INPUT_TEXT = "พิมพ์ข้อความงานของคุณที่นี่..."

# ✏️ จุดที่ 2: ปรับกติกา หมวดหมู่ และชื่อ field ให้เข้ากับงานของคุณ
PROMPT = f"""
คุณเป็นผู้ช่วย AI ของแผนก ______ ที่ BKC
วิเคราะห์ข้อความด้านล่าง แล้วตอบกลับเป็น JSON เท่านั้น
field ที่ต้องการ: summary, category, priority, next_action

ข้อความ:
{INPUT_TEXT}
"""

try:
    r = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You return valid JSON only."},
            {"role": "user", "content": PROMPT},
        ],
        response_format={"type": "json_object"},
    )
    result = json.loads(r.choices[0].message.content)
    print("📋 ผลลัพธ์:")
    for k, v in result.items():
        print(f" {k}: {v}")
except Exception as e:
    print("⚠️ เกิดข้อผิดพลาด รอแล้วกด Run ใหม่ หรือเรียก TA:", e)


---
### 🎉 จบ Workshop!

- สิ่งที่ทำได้ตอนนี้: เรียก AI ผ่านโค้ด, บังคับให้ตอบ JSON, ทำ Human-in-the-Loop, ประมวลผลเป็นชุดลง Excel
- ทักษะที่ติดตัวกลับไป: **การเขียน Prompt** (Role / Task / Rules / Output) ใช้ต่อกับ ChatGPT/Copilot ได้เลย
- ก้าวต่อไป: RAG (ให้ AI ตอบจากคู่มือบริษัท), M365 Copilot / Power Platform
